In [ ]:
# !conda create -n skmob python=3.9.0
# !conda activate skmob
# !conda install conda-forge scikit-mobility

In [1]:
import networkx as nx
import pandas as pd # version 2.3.1
import geopandas as gpd #1.0.1
import numpy as np # 2.2.6
import shapely   # 2.0.7 , measure importance of features through SHAP values
import skmob

In [2]:
# large data files prevent sync on VS Code, so connect with dropbox
# !conda install conda-forge::dropbox
import dropbox
import os
# %pip install dotenv
from dotenv import load_dotenv  # use load_dotenv for dropbox secrets storage

Create a file named '.env' in project directory and write the following lines:
- DROPBOX_APP_KEY= <your_app_key_here>
- DROPBOX_APP_SECRET= <your_app_secret_here>

See 'Python use env file.png' to allow terminal access to .env file

In [ ]:
# https://www.dropbox.com/developers/documentation/python#tutorial
from dropbox import DropboxOAuth2FlowNoRedirect

load_dotenv()
APP_KEY = os.environ.get("DROPBOX_APP_KEY")
APP_SECRET = os.environ.get("DROPBOX_APP_SECRET")

auth_flow = DropboxOAuth2FlowNoRedirect(APP_KEY,
                                        use_pkce = True,
                                        consumer_secret=APP_SECRET,
                                        token_access_type='offline',
                                        scope=['files.metadata.read'])

authorize_url = auth_flow.start()
print("1. Go to: " + authorize_url)
print("2. Click \"Allow\" (you might have to log in first).")
print("3. Copy the authorization code.")
auth_code = input("Enter the authorization code here: ").strip()

try:
    oauth_result = auth_flow.finish(auth_code)
    assert oauth_result.scope == 'files.metadata.read'
except Exception as e:
    print('Error: %s' % (e,))
    exit(1)

with dropbox.Dropbox(oauth2_access_token=oauth_result.access_token,
                     oauth2_access_token_expiration=oauth_result.expires_at,
                     oauth2_refresh_token=oauth_result.refresh_token,
                     app_key=APP_KEY,
                     app_secret=APP_SECRET):
    print("Successfully set up client!")

1. Go to: https://www.dropbox.com/oauth2/authorize?response_type=code&client_id=6d2y8zfqqivigkd&token_access_type=offline&code_challenge=udW5x0kfu2y3CQo9mKb8Hp8dSCyvqDTioQinV1c9Lx8&code_challenge_method=S256&scope=files.metadata.read
2. Click "Allow" (you might have to log in first).
3. Copy the authorization code.
Successfully set up client!


Make sure that there is data within the app folder created in the user's DropBox. (The app does not automatically copy files from the DropBox folder it was created from).

In [12]:
dbx = dropbox.Dropbox(oauth2_access_token=oauth_result.access_token,
                     oauth2_access_token_expiration=oauth_result.expires_at,
                     oauth2_refresh_token=oauth_result.refresh_token,
                     app_key=APP_KEY,
                     app_secret=APP_SECRET)

In [ ]:
for entry in dbx.files_list_folder('').entries:
    print(entry.name)

2017-2021 (1)


In [29]:
folder_path = dbx.files_list_folder('').entries[0]

BadInputError: BadInputError('40e8558745f1417192427a1e69c58747', 'Error in call to API function "files/list_folder": This app is currently disabled.')

In [ ]:
data_folder = dbx.files_list_folder(folder_path).entries

NameError: name 'dropbox_folder_path' is not defined

In [ ]:
from io import BytesIO

In [ ]:
# datasets is aggregated commuter trajectories
datasets = {}
for file_name in data_folder:
    metaData, data = dbx.files_download(path)
    df = pd.read_csv(BytesIO(data.content)
    # file_name.split("_") creates list of items split from the filename -> [-1] access last item -> .split(".") splits last item by "."-> [0] accesses the first item which is the state code
    state_code = file_name.split("_")[-1].split(".")[0]
    file_path = os.path.join(data_folder, file_name)
    df = pd.read_csv(file_path)
    # Add the DataFrame to the container with the state code as the label
    datasets[state_code] = df

In [ ]:
data_folder = "data/ctpp/2017-2021"

for filename in os.listdir(data_folder):
    # filename.split("_") creates list of items split from the filename -> [-1] access last item -> .split(".") splits last item by "."-> [0] accesses the first item which is the state code
    state_code = filename.split("_")[-1].split(".")[0]
    file_path = os.path.join(data_folder, filename)
    df = pd.read_csv(file_path)
    # Add the DataFrame to the container with the state code as the label
    datasets[state_code] = df
    

In [33]:
datasets['FL'].groupby("Otract").nunique()

,Dtract,EST
Otract,,
12001000201,24,15
12001000202,28,15
12001000301,27,19
12001000302,27,13
12001000400,28,16
...,...,...
12133970104,14,10
12133970200,27,10
12133970301,25,12


In [6]:
len(datasets['FL']['Dtract'].unique())

8253

In [4]:
len(datasets['FL']['Otract'].unique())

5075

In [7]:
set(datasets['FL']['Otract'].unique()) - set(datasets['FL']['Dtract'].unique())

{np.int64(12071010107), np.int64(12083000806), np.int64(12099005943)}

In [ ]:
set(datasets['FL']['Dtract'].unique()) - set(datasets['FL']['Otract'].unique())
# len(set(datasets['FL']['Dtract'].unique()) - set(datasets['FL']['Otract'].unique()))

{np.int64(36071014400),
 np.int64(6025011201),
 np.int64(17197883402),
 np.int64(51700031500),
 np.int64(19153010707),
 np.int64(13051002900),
 np.int64(48491020310),
 np.int64(13081010201),
 np.int64(36011040800),
 np.int64(13127000101),
 np.int64(13127000103),
 np.int64(34013020200),
 np.int64(34017017900),
 np.int64(19113001005),
 np.int64(34039038000),
 np.int64(17031831600),
 np.int64(45063020601),
 np.int64(45063020605),
 np.int64(36047011901),
 np.int64(48491020352),
 np.int64(21111980100),
 np.int64(39153532999),
 np.int64(42101014600),
 np.int64(36103135303),
 np.int64(48113016903),
 np.int64(36119011401),
 np.int64(51107610702),
 np.int64(17043841102),
 np.int64(27147960400),
 np.int64(6085509201),
 np.int64(37083930702),
 np.int64(17043841111),
 np.int64(9001050200),
 np.int64(26125197400),
 np.int64(36047044700),
 np.int64(35013001309),
 np.int64(17037000800),
 np.int64(13095010401),
 np.int64(29189218401),
 np.int64(12009980000),
 np.int64(1097007204),
 np.int64(3400304240

In [15]:
datasets['FL']

,Otract,Dtract,EST
0,12001000201,12001000201,405.0
1,12001000201,12001000202,40.0
2,12001000201,12001000301,20.0
3,12001000201,12001000400,225.0
4,12001000201,12001000500,35.0
...,...,...,...
220720,12133970303,12133970103,55.0
220721,12133970303,12133970104,105.0
220722,12133970303,12133970200,15.0
220723,12133970303,12133970302,50.0


In [17]:
# datasets['FL']['Otract' = np.int64(12071010107)]['EST']
datasets['FL'].loc[datasets['FL']['Otract'] == 12071010107]

,Otract,Dtract,EST
83194,12071010107,2020002301,30.0
83195,12071010107,12015010301,60.0
83196,12071010107,12015010302,10.0
83197,12071010107,12015020303,4.0
83198,12071010107,12071000600,15.0
83199,12071010107,12071000700,30.0
83200,12071010107,12071000800,20.0
83201,12071010107,12071001104,15.0
83202,12071010107,12071001203,20.0
83203,12071010107,12071001207,25.0


In [18]:
datasets['FL'].loc[datasets['FL']['Dtract'] == 12071010107]

,Otract,Dtract,EST


Some census tracts are listed only as an origin and not as a destination census tract. This is interpreted as there being no commuter workers within those census tracts. Additionally, some census tracts are listed only as destinations and not as origin census tracts. This is interpreted as the resident population is 0. 

We will calculate the Population of a census tract as the sum of travelers when grouped by an Otract id matching the census tract id.
The Number of Jobs held by commuters is the sum of travelers when grouped by a Dtract id matching the census tract id. For 

In [44]:
grouped_dsets = {}

In [73]:
fl_data = datasets['FL']
no_jobs = pd.DataFrame(set(fl_data['Otract'].unique()) - set(fl_data['Dtract'].unique()), columns = ['Id'])
no_pop = pd.DataFrame(set(fl_data['Dtract'].unique()) - set(fl_data['Otract'].unique()), columns = ['Id'])

In [85]:
grouped = pd.DataFrame()
grouped['Id'] = df['Dtract'].unique()
grouped = pd.concat([grouped, no_jobs])

In [86]:
len(grouped)

367

In [ ]:
grouped.loc[grouped['Id'].isin(no_jobs['Id']), 'Jobs'] = 0
jobs = df.groupby('Dtract')['EST'].sum()
grouped['Id'] = grouped['Id'].map(jobs)        #error with this line

In [88]:
len(grouped)

367

In [89]:
grouped

,Id,Jobs
0,724.0,NaN
1,738.0,NaN
2,263.0,NaN
3,1120.0,NaN
4,623.0,NaN
...,...,...
362,40.0,NaN
363,30.0,NaN
0,NaN,0.0
1,NaN,0.0


In [49]:
for state_code, df in datasets.items():
    grouped = pd.DataFrame()
    # no_jobs is a datafram of tracts that have a resident population, but don't provide any jobs for commuting workers from other census tracts
    no_jobs = pd.DataFrame(set(df['Otract'].unique()) - set(df['Dtract'].unique()), columns = ['Id'])
    # no_pop is a data frame of tracts that provide jobs,  but do not have any resident population
    no_pop = pd.DataFrame(set(df['Dtract'].unique()) - set(df['Otract'].unique()), columns = ['Id'])
    
    grouped['Id'] = df['Dtract'].unique()
    # now add the Ids of the census tracts that are only origin tracts and don't function as destination tract
    grouped = pd.concat([grouped, no_jobs])

    # Now add null job values to census tracts that are only origin tracts
    grouped.loc[grouped['Id'].isin(no_jobs['Id']), 'Jobs'] = 0
    # add job values to all other tracts with jobs
    jobs = df.groupby('Dtract')['EST'].sum()
    grouped['Id'].map(jobs)

    # Add null pop values to census tracts that are only destination tracts
    grouped.loc[grouped['Id'].isin(no_pop['Id']), 'Population'] = 0
    # add pop values to all other tracts with pop
    pop = df.groupby('Otract')['EST'].sum()
    grouped['Population'].map(pop)
    grouped_dsets[state_code] = grouped

In [64]:
assert not grouped_dsets['FL'].isna().values.any()

AssertionError: 

In [55]:
fl_data = grouped_dsets['FL']

In [72]:
fl_data[fl_data.notna().all(axis=1)]

,Id,Jobs,Population


In [62]:
fl_data[fl_data["Population"] == 0]

,Id,Jobs,Population
37,1073002402,NaN,0.0
57,48163950301,NaN,0.0
79,37063002038,NaN,0.0
85,21111003501,NaN,0.0
86,21111011007,NaN,0.0
...,...,...,...
8247,17043846004,NaN,0.0
8248,1103000400,NaN,0.0
8249,13097080203,NaN,0.0
8250,1101003301,NaN,0.0


### Now add other predictors used by the traditional gravity model: :
- longitude, latitude of the centroid of each census tract


In [ ]:
# geopandas, read the TIGER shapefile

### Now add additional predictors to use:
- SVI info for each census tract
- POI_lon, POI_lat where POI is the some POI for each census tract
- POI_text which is textual data about the rating, review of the POI